In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import time
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    GPT2Tokenizer, GPT2LMHeadModel,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    LongformerTokenizer, LongformerForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories for models and results
Path('models').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)

Using device: cuda


In [ ]:
BASELINE_LM_HYPERPARAMS = {
    'RNN': {
        'vocab_size': 50257,
        'embedding_dim': 256,
        'hidden_dim': 512,
        'num_layers': 2,
        'dropout': 0.3,
        'learning_rate': 5e-4,
        'batch_size': 32,
        'num_epochs': 3,
        'max_length': 256,
        'weight_decay': 0.01
    },
    'LSTM': {
        'vocab_size': 50257,
        'embedding_dim': 256,
        'hidden_dim': 512,
        'num_layers': 2,
        'dropout': 0.3,
        'learning_rate': 3e-4,
        'batch_size': 32,
        'num_epochs': 3,
        'max_length': 256,
        'weight_decay': 0.01
    },
    'GRU': {
        'vocab_size': 50257,
        'embedding_dim': 256,
        'hidden_dim': 512,
        'num_layers': 2,
        'dropout': 0.3,
        'learning_rate': 3e-4,
        'batch_size': 32,
        'num_epochs': 3,
        'max_length': 256,
        'weight_decay': 0.01
    }
}

MINIMAL_LM_HYPERPARAMS = {
    'minLSTM': {
        'vocab_size': 50257,
        'embedding_dim': 256,
        'hidden_dim': 512,
        'num_layers': 3,
        'dropout': 0.2,
        'learning_rate': 1e-3,
        'batch_size': 96,
        'num_epochs': 5,
        'max_length': 128,
        'weight_decay': 0.01
    },
    'minGRU': {
        'vocab_size': 50257,
        'embedding_dim': 256,
        'hidden_dim': 512,
        'num_layers': 3,
        'dropout': 0.2,
        'learning_rate': 1.5e-3,
        'batch_size': 64,
        'num_epochs': 5,
        'max_length': 128,
        'weight_decay': 0.01
    }
}

TRANSFORMER_LM_HYPERPARAMS = {
    'GPT2-Small': {
        'model_name': 'gpt2',
        'learning_rate': 5e-5,
        'batch_size': 4,
        'num_epochs': 3,
        'max_length': 256,
        'warmup_steps': 200,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 8
    }
}

BASELINE_CLF_HYPERPARAMS = {
    'RNN': {
        'vocab_size': 30522,
        'embedding_dim': 200,
        'hidden_dim': 256,
        'num_classes': 4,
        'num_layers': 3,
        'dropout': 0.3,
        'learning_rate': 1e-4,
        'batch_size': 64,
        'num_epochs': 3,
        'max_length': 2128,
        'weight_decay': 0.01
    },
    'LSTM': {
        'vocab_size': 30522,
        'embedding_dim': 200,
        'hidden_dim': 256,
        'num_classes': 4,
        'num_layers': 2,
        'dropout': 0.4,
        'learning_rate': 1e-3,
        'batch_size': 64,
        'num_epochs': 5,
        'max_length': 256,
        'weight_decay': 0.01
    },
    'GRU': {
        'vocab_size': 30522,
        'embedding_dim': 200,
        'hidden_dim': 256,
        'num_classes': 4,
        'num_layers': 2,
        'dropout': 0.4,
        'learning_rate': 1e-3,
        'batch_size': 64,
        'num_epochs': 5,
        'max_length': 256,
        'weight_decay': 0.01
    }
}

MINIMAL_CLF_HYPERPARAMS = {
    'minLSTM': {
        'vocab_size': 30522,
        'embedding_dim': 300,
        'hidden_dim': 512,
        'num_classes': 4,
        'num_layers': 3,
        'dropout': 0.3,
        'learning_rate': 1.2e-3,
        'batch_size': 64,
        'num_epochs': 3,
        'max_length': 256,
        'weight_decay': 0.01
    },
    'minGRU': {
        'vocab_size': 30522,
        'embedding_dim': 300,
        'hidden_dim': 512,
        'num_classes': 4,
        'num_layers': 3,
        'dropout': 0.3,
        'learning_rate': 1.2e-3,
        'batch_size': 64,
        'num_epochs': 3,
        'max_length': 256,
        'weight_decay': 0.01
    }
}

TRANSFORMER_CLF_HYPERPARAMS = {
    'DistilBERT': {
        'model_name': 'distilbert-base-uncased',
        'learning_rate': 3e-5,
        'batch_size': 16,
        'num_epochs': 3,
        'max_length': 256,
        'warmup_steps': 500,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 2
    },
    'Longformer': {
        'model_name': 'allenai/longformer-base-4096',
        'learning_rate': 2e-5,
        'batch_size': 8,
        'num_epochs': 3,
        'max_length': 512,
        'attention_window': 512,
        'warmup_steps': 500,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 4
    }
}

print("\n=== Hyperparameter Configuration Loaded ===")
print(f"Language Modeling - Baseline Models: {list(BASELINE_LM_HYPERPARAMS.keys())}")
print(f"Language Modeling - Minimal Models: {list(MINIMAL_LM_HYPERPARAMS.keys())}")
print(f"Language Modeling - Transformers: {list(TRANSFORMER_LM_HYPERPARAMS.keys())}")
print(f"Classification - Baseline Models: {list(BASELINE_CLF_HYPERPARAMS.keys())}")
print(f"Classification - Minimal Models: {list(MINIMAL_CLF_HYPERPARAMS.keys())}")
print(f"Classification - Transformers: {list(TRANSFORMER_CLF_HYPERPARAMS.keys())}")



=== Hyperparameter Configuration Loaded ===
Language Modeling - Baseline Models: ['RNN', 'LSTM', 'GRU']
Language Modeling - Minimal Models: ['minLSTM', 'minGRU']
Language Modeling - Transformers: ['GPT2-Small']
Classification - Baseline Models: ['RNN', 'LSTM', 'GRU']
Classification - Minimal Models: ['minLSTM', 'minGRU']
Classification - Transformers: ['DistilBERT', 'Longformer']


In [ ]:
class ModelManager:
    def __init__(self, model_dir='models'):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(exist_ok=True)

    def save_model(self, model, optimizer, config, model_name, epoch, metrics=None):
        model_path = self.model_dir / f"{model_name}_epoch{epoch}.pt"

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': config,
            'metrics': metrics or {},
            'timestamp': datetime.now().isoformat()
        }

        torch.save(checkpoint, model_path)
        print(f"✓ Model saved: {model_path}")

        config_path = self.model_dir / f"{model_name}_config.json"
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)

        return model_path

    def load_model(self, model_class, model_name, epoch=None, device='cpu'):
        if epoch is None:
            checkpoints = list(self.model_dir.glob(f"{model_name}_epoch*.pt"))
            if not checkpoints:
                raise FileNotFoundError(f"No checkpoints found for {model_name}")
            model_path = max(checkpoints, key=lambda p: int(p.stem.split('epoch')[-1]))
        else:
            model_path = self.model_dir / f"{model_name}_epoch{epoch}.pt"

        if not model_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {model_path}")

        checkpoint = torch.load(model_path, map_location=device)
        config = checkpoint['config']

        model = model_class(**config).to(device)
        model.load_state_dict(checkpoint['model_state_dict'])

        optimizer = optim.AdamW(model.parameters(), lr=config.get('learning_rate', 1e-4))
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        print(f"✓ Model loaded: {model_path}")
        print(f" Config: {config}")
        print(f" Metrics: {checkpoint['metrics']}")

        return model, optimizer, config, checkpoint['metrics'], checkpoint['epoch']

    def list_saved_models(self):
        checkpoints = list(self.model_dir.glob("*.pt"))
        if not checkpoints:
            print("No saved models found.")
            return

        print("Saved Models:")
        for cp in sorted(checkpoints):
            print(f" - {cp.name}")


model_manager = ModelManager()

In [ ]:
from collections import Counter

class CustomVocabBuilder:
    def __init__(self, vocab_size=50257):
        self.vocab_size = vocab_size
        self.word_counts = Counter()

    def build_vocab_from_dataset(self, dataset, text_field='text'):
        print(f"Building vocabulary from {len(dataset)} samples...")
        for sample in tqdm(dataset, desc="Counting words"):
            text = sample[text_field].lower()
            words = text.split()
            self.word_counts.update(words)

        most_common = self.word_counts.most_common(self.vocab_size - 4)

        vocab = {
            '<pad>': 0,
            '<unk>': 1,
            '<bos>': 2,
            '<eos>': 3
        }

        for idx, (word, _) in enumerate(most_common, start=4):
            vocab[word] = idx

        print(f"Vocabulary built: {len(vocab)} tokens")
        return vocab

    def create_tokenizer(self, vocab):
        id_to_token = {v: k for k, v in vocab.items()}

        vocab_file = 'custom_vocab.json'
        with open(vocab_file, 'w') as f:
            json.dump(vocab, f)

        tokenizer = CustomTokenizer(vocab, id_to_token)
        return tokenizer


class CustomTokenizer:
    def __init__(self, vocab, id_to_token):
        self.vocab = vocab
        self.id_to_token = id_to_token
        self.vocab_size = len(vocab)
        self.pad_token = '<pad>'
        self.pad_token_id = vocab['<pad>']
        self.unk_token = '<unk>'
        self.unk_token_id = vocab['<unk>']

    def __call__(self, text, max_length=256, padding='max_length',
                 truncation=True, return_tensors='pt'):
        if isinstance(text, list):
            return self.batch_encode(text, max_length, padding, truncation, return_tensors)

        words = text.lower().split()
        token_ids = [self.vocab.get(word, self.unk_token_id) for word in words]

        if truncation and len(token_ids) > max_length:
            token_ids = token_ids[:max_length]

        if padding == 'max_length':
            attention_mask = [1] * len(token_ids) + [0] * (max_length - len(token_ids))
            token_ids = token_ids + [self.pad_token_id] * (max_length - len(token_ids))
        else:
            attention_mask = [1] * len(token_ids)

        if return_tensors == 'pt':
            return {
                'input_ids': torch.tensor([token_ids]),
                'attention_mask': torch.tensor([attention_mask])
            }
        return {'input_ids': token_ids, 'attention_mask': attention_mask}

    def batch_encode(self, texts, max_length, padding, truncation, return_tensors):
        all_input_ids = []
        all_attention_masks = []

        for text in texts:
            encoded = self(text, max_length, padding, truncation, return_tensors=None)
            all_input_ids.append(encoded['input_ids'])
            all_attention_masks.append(encoded['attention_mask'])

        if return_tensors == 'pt':
            return {
                'input_ids': torch.tensor(all_input_ids),
                'attention_mask': torch.tensor(all_attention_masks)
            }
        return {'input_ids': all_input_ids, 'attention_mask': all_attention_masks}


In [ ]:
class WikiTextStreamingDataset(Dataset):
    def __init__(self, tokenizer, max_length=512, split='train', cache_dir='./cache'):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.split = split

        print(f"Loading WikiText-103 {split} split...")
        self.dataset = load_dataset('wikitext', 'wikitext-103-v1', split=split, cache_dir=cache_dir)
        self.dataset = self.dataset.filter(lambda x: len(x['text'].strip()) > 0)
        print(f"Dataset size: {len(self.dataset)} samples")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]['text']

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


def create_language_modeling_dataloaders(
    tokenizer, batch_size=32, max_length=512,
    num_workers=0, cache_dir='./cache'
):
    splits = {'train': 'train', 'validation': 'validation', 'test': 'test'}
    dataloaders = {}

    for split_name, split_type in splits.items():
        dataset = WikiTextStreamingDataset(
            tokenizer=tokenizer,
            max_length=max_length,
            split=split_type,
            cache_dir=cache_dir
        )

        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split_type == 'train'),
            num_workers=num_workers,
            pin_memory=True
        )

        dataloaders[split_name] = dataloader

    return dataloaders


class AGNewsStreamingDataset(Dataset):
    def __init__(self, tokenizer, max_length=256, split='train', cache_dir='./cache'):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.split = split

        print(f"Loading AG News {split} split...")
        self.dataset = load_dataset('ag_news', split=split, cache_dir=cache_dir)
        print(f"Dataset size: {len(self.dataset)} samples")

        self.class_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Tech'}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        text = sample['text']
        label = sample['label']

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': torch.tensor(label, dtype=torch.long)
        }


def create_classification_dataloaders(
    tokenizer, batch_size=32, max_length=256,
    num_workers=0, cache_dir='./cache'
):
    train_dataset = AGNewsStreamingDataset(
        tokenizer=tokenizer,
        max_length=max_length,
        split='train',
        cache_dir=cache_dir
    )

    test_dataset = AGNewsStreamingDataset(
        tokenizer=tokenizer,
        max_length=max_length,
        split='test',
        cache_dir=cache_dir
    )

    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size

    train_subset, val_subset = random_split(
        train_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        rnn_out, _ = self.rnn(x)
        logits = self.fc(rnn_out)
        return logits


class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits


class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        gru_out, _ = self.gru(x)
        logits = self.fc(gru_out)
        return logits


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        rnn_out, _ = self.rnn(x)
        last_out = rnn_out[:, -1, :]
        logits = self.fc(last_out)
        return logits


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        lstm_out, (h_n, _) = self.lstm(x)
        x = torch.cat([h_n[-2, :, :], h_n[-1, :, :]], dim=1)
        logits = self.fc(x)
        return logits


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.dropout_rate = dropout

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        gru_out, h_n = self.gru(x)
        x = torch.cat([h_n[-2, :, :], h_n[-1, :, :]], dim=1)
        logits = self.fc(x)
        return logits


In [ ]:
class minLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.dropout_rate = dropout
        self.vocab_size = vocab_size

        self.input_proj = nn.Linear(embedding_dim, hidden_dim * 2)

        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim * 4) for _ in range(num_layers)
        ])

        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        batch_size, seq_len, _ = x.size()

        x = self.input_proj(x)
        h, c = x.chunk(2, dim=-1)

        for layer_idx in range(self.num_layers):
            gates = self.gates[layer_idx]

            gate_vals = gates(h)
            i_t, f_t, o_t, g_t = gate_vals.chunk(4, dim=-1)

            i_t = torch.sigmoid(i_t)
            f_t = torch.sigmoid(f_t)
            o_t = torch.sigmoid(o_t)
            g_t = torch.tanh(g_t)

            c = f_t * c + i_t * g_t
            h = o_t * torch.tanh(c)

        logits = self.output_proj(h)
        return logits


class minGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.dropout_rate = dropout
        self.vocab_size = vocab_size

        self.input_proj = nn.Linear(embedding_dim, hidden_dim)

        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim * 3) for _ in range(num_layers)
        ])

        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        h = self.input_proj(x)

        for layer_idx in range(self.num_layers):
            gates = self.gates[layer_idx]

            gate_vals = gates(h)
            r_t, z_t, n_t = gate_vals.chunk(3, dim=-1)

            r_t = torch.sigmoid(r_t)
            z_t = torch.sigmoid(z_t)
            n_t = torch.tanh(n_t)

            h = (1 - z_t) * n_t + z_t * h

        logits = self.output_proj(h)
        return logits


class minLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.dropout_rate = dropout

        self.input_proj = nn.Linear(embedding_dim, hidden_dim * 2)
        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim * 4) for _ in range(num_layers)
        ])
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        x = self.input_proj(x)
        h, c = x.chunk(2, dim=-1)

        for layer_idx in range(self.num_layers):
            gates = self.gates[layer_idx]
            gate_vals = gates(h)
            i_t, f_t, o_t, g_t = gate_vals.chunk(4, dim=-1)

            i_t = torch.sigmoid(i_t)
            f_t = torch.sigmoid(f_t)
            o_t = torch.sigmoid(o_t)
            g_t = torch.tanh(g_t)

            c = f_t * c + i_t * g_t
            h = o_t * torch.tanh(c)

        h_pooled = h.mean(dim=1)
        logits = self.fc(h_pooled)
        return logits


class minGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, dropout=0.3, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.dropout_rate = dropout

        self.input_proj = nn.Linear(embedding_dim, hidden_dim)
        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim * 3) for _ in range(num_layers)
        ])
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        h = self.input_proj(x)

        for layer_idx in range(self.num_layers):
            gates = self.gates[layer_idx]
            gate_vals = gates(h)
            r_t, z_t, n_t = gate_vals.chunk(3, dim=-1)

            r_t = torch.sigmoid(r_t)
            z_t = torch.sigmoid(z_t)
            n_t = torch.tanh(n_t)

            h = (1 - z_t) * n_t + z_t * h

        h_pooled = h.mean(dim=1)
        logits = self.fc(h_pooled)
        return logits


In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train_lm_epoch(model, dataloader, optimizer, device, scheduler):
    model.train()
    total_loss = 0
    total_tokens = 0
    scaler = GradScaler(device="cuda")

    start_time = time.time()

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        with autocast():
            logits = model(input_ids, attention_mask)

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)

            loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_tokens += (shift_labels != -100).sum().item()

    elapsed_time = time.time() - start_time
    avg_loss = total_loss / len(dataloader)
    tokens_per_sec = total_tokens / elapsed_time

    return avg_loss, tokens_per_sec


def evaluate_lm(model, dataloader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            with autocast():
                logits = model(input_ids, attention_mask)

                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels[:, 1:].contiguous()

                shift_logits = shift_logits.view(-1, shift_logits.size(-1))
                shift_labels = shift_labels.view(-1)

                loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)

            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    perplexity = torch.exp(torch.tensor(avg_loss)).item()

    return avg_loss, perplexity



def measure_efficiency(model, dataloader, device, num_batches=5):
    model.eval()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    total_samples = 0

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_batches:
                break

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch.get('attention_mask')
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)

            with autocast(device="cuda"):
                _ = model(input_ids, attention_mask)

            total_samples += input_ids.size(0)

    elapsed_time = time.time() - start_time

    metrics = {
        'samples_per_sec': total_samples / elapsed_time,
        'ms_per_sample': (elapsed_time / total_samples) * 1000
    }

    if torch.cuda.is_available():
        metrics['peak_memory_mb'] = torch.cuda.max_memory_allocated() / 1024**2

    return metrics


In [ ]:
def train_rnn_language_model(device, hyperparams=None, use_custom_vocab=True):
    if hyperparams is None:
        hyperparams = BASELINE_LM_HYPERPARAMS['RNN'].copy()
    if use_custom_vocab:
        hyperparams['vocab_size'] = 50257

    model_name = 'rnn_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    if use_custom_vocab:
        raw_dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train[:50000]', cache_dir='./cache')
        vocab_builder = CustomVocabBuilder(vocab_size=50257)
        vocab = vocab_builder.build_vocab_from_dataset(raw_dataset)
        tokenizer = vocab_builder.create_tokenizer(vocab)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = RNNLanguageModel(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'train_tokens_per_sec': []
    }
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, tokens_per_sec = train_lm_epoch(
            model, dataloaders['train'], optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)
        print(f"Train Loss: {train_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        val_loss, val_perplexity = evaluate_lm(model, dataloaders['validation'], device)
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)
        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_loss': val_loss, 'val_perplexity': val_perplexity}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_perplexity = evaluate_lm(model, dataloaders['test'], device)
    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


def train_lstm_language_model(device, hyperparams=None, use_custom_vocab=True):
    if hyperparams is None:
        hyperparams = BASELINE_LM_HYPERPARAMS['LSTM'].copy()
        if use_custom_vocab:
            hyperparams['vocab_size'] = 50257

    model_name = 'lstm_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    if use_custom_vocab:
        raw_dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train[:50000]', cache_dir='./cache')
        vocab_builder = CustomVocabBuilder(vocab_size=50257)
        vocab = vocab_builder.build_vocab_from_dataset(raw_dataset)
        tokenizer = vocab_builder.create_tokenizer(vocab)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = LSTMLanguageModel(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'val_loss': [], 'val_perplexity': [], 'train_tokens_per_sec': []}
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, tokens_per_sec = train_lm_epoch(
            model, dataloaders['train'], optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)
        print(f"Train Loss: {train_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        val_loss, val_perplexity = evaluate_lm(model, dataloaders['validation'], device)
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)
        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_loss': val_loss, 'val_perplexity': val_perplexity}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_perplexity = evaluate_lm(model, dataloaders['test'], device)
    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency

def train_gru_language_model(device, hyperparams=None, use_custom_vocab=True):
    """Train GRU language model."""
    if hyperparams is None:
        hyperparams = BASELINE_LM_HYPERPARAMS['GRU'].copy()
        if use_custom_vocab:
            hyperparams['vocab_size'] = 50257

    model_name = 'gru_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    # Tokenizer - Custom or GPT2
    if use_custom_vocab:
        raw_dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train[:50000]', cache_dir='./cache')
        vocab_builder = CustomVocabBuilder(vocab_size=50257)
        vocab = vocab_builder.build_vocab_from_dataset(raw_dataset)
        tokenizer = vocab_builder.create_tokenizer(vocab)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = GRULanguageModel(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'val_loss': [], 'val_perplexity': [], 'train_tokens_per_sec': []}
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, tokens_per_sec = train_lm_epoch(
            model, dataloaders['train'], optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)
        print(f"Train Loss: {train_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        val_loss, val_perplexity = evaluate_lm(model, dataloaders['validation'], device)
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)
        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_loss': val_loss, 'val_perplexity': val_perplexity}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_perplexity = evaluate_lm(model, dataloaders['test'], device)
    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency

In [ ]:
def train_minlstm_language_model(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = MINIMAL_LM_HYPERPARAMS['minLSTM']

    model_name = 'minlstm_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model (OPTIMIZED)")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length'],
        num_workers=0,
        cache_dir='./cache'
    )

    model = minLSTM(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=hyperparams['learning_rate'],
        weight_decay=hyperparams['weight_decay']
    )

    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'train_tokens_per_sec': []
    }
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")

        train_loss, tokens_per_sec = train_lm_epoch(
            model, dataloaders['train'], optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)
        print(f"Train Loss: {train_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        val_loss, val_perplexity = evaluate_lm(
            model, dataloaders['validation'], device
        )
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)
        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'val_perplexity': val_perplexity
                }
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_perplexity = evaluate_lm(model, dataloaders['test'], device)
    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


def train_mingru_language_model(device, hyperparams=None):
    """Train minGRU language model with optimizations."""
    if hyperparams is None:
        hyperparams = MINIMAL_LM_HYPERPARAMS['minGRU']

    model_name = 'mingru_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model (OPTIMIZED)")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length'],
        num_workers=0,
        cache_dir='./cache'
    )

    model = minGRU(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=hyperparams['learning_rate'],
        weight_decay=hyperparams['weight_decay']
    )

    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'train_tokens_per_sec': []
    }
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")

        train_loss, tokens_per_sec = train_lm_epoch(
            model, dataloaders['train'], optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)
        print(f"Train Loss: {train_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        val_loss, val_perplexity = evaluate_lm(
            model, dataloaders['validation'], device
        )
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)
        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'val_perplexity': val_perplexity
                }
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_perplexity = evaluate_lm(model, dataloaders['test'], device)
    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


In [ ]:
def train_gpt2_language_model(device, hyperparams=None, model_manager=None,
                                create_language_modeling_dataloaders=None):
    if hyperparams is None:
        hyperparams = TRANSFORMER_LM_HYPERPARAMS['GPT2-Small']

    model_name = 'gpt2_small_lm'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Language Model")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

    dataloaders = create_language_modeling_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = AdamW(model.parameters(),
                     lr=hyperparams['learning_rate'],
                     weight_decay=hyperparams['weight_decay'])

    total_steps = len(dataloaders['train']) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=hyperparams['warmup_steps'],
        num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'train_tokens_per_sec': []
    }
    best_val_perplexity = float('inf')

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")

        model.train()
        total_loss = 0
        total_tokens = 0
        start_time = time.time()

        for batch_idx, batch in enumerate(tqdm(dataloaders['train'], desc="Training")):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, labels=labels, return_dict=True)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * input_ids.size(0)
            total_tokens += input_ids.numel()

        epoch_time = time.time() - start_time
        avg_loss = total_loss / total_tokens
        tokens_per_sec = total_tokens / epoch_time

        history['train_loss'].append(avg_loss)
        history['train_tokens_per_sec'].append(tokens_per_sec)

        print(f"Train Loss: {avg_loss:.4f} | Tokens/sec: {tokens_per_sec:.0f}")

        model.eval()
        total_val_loss = 0
        total_val_tokens = 0

        with torch.no_grad():
            for batch in tqdm(dataloaders['validation'], desc="Validating"):
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids, labels=labels, return_dict=True)
                loss = outputs.loss

                total_val_loss += loss.item() * input_ids.size(0)
                total_val_tokens += input_ids.numel()

        val_loss = total_val_loss / total_val_tokens
        val_perplexity = np.exp(val_loss)

        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_perplexity)

        print(f"Val Loss: {val_loss:.4f} | Perplexity: {val_perplexity:.4f}\n")

        if val_perplexity < best_val_perplexity:
            best_val_perplexity = val_perplexity
            model_path = Path('models') / f"{model_name}_epoch{epoch}.pt"
            torch.save(model.state_dict(), model_path)
            print(f"✓ Model saved: {model_path}")

    print(f"\n{'='*80}")
    print("Testing...")
    model.eval()
    total_test_loss = 0
    total_test_tokens = 0

    with torch.no_grad():
        for batch in tqdm(dataloaders['test'], desc="Testing"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, labels=labels, return_dict=True)
            loss = outputs.loss

            total_test_loss += loss.item() * input_ids.size(0)
            total_test_tokens += input_ids.numel()

    test_loss = total_test_loss / total_test_tokens
    test_perplexity = np.exp(test_loss)

    print(f"Test Loss: {test_loss:.4f} | Test Perplexity: {test_perplexity:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, dataloaders['test'], device, num_batches=5)
    for key, val in efficiency.items():
        print(f"  {key}: {val:.2f}")

    return model, history, efficiency

In [ ]:
def train_classification_epoch(model, dataloader, optimizer, device, scheduler):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    scaler = GradScaler()

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        with autocast():
            logits = model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    return avg_loss, accuracy, f1, all_preds


def evaluate_classification(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            with autocast():
                logits = model(input_ids, attention_mask)
                loss = F.cross_entropy(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted',zero_division=0)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    print("\n" + classification_report(all_labels, all_preds,
                                   target_names=['World', 'Sports', 'Business', 'Tech'],
                                   zero_division=0))

    return avg_loss, accuracy, f1, precision, recall

In [ ]:
def train_rnn_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = BASELINE_CLF_HYPERPARAMS['RNN']

    model_name = 'rnn_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = RNNClassifier(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_classes=hyperparams['num_classes'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
               'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, train_acc, train_f1, _ = train_classification_epoch(
            model, train_loader, optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        val_loss, val_acc, val_f1, _, _ = evaluate_classification(model, val_loader, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_acc': val_acc, 'val_f1': val_f1}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_acc, test_f1, test_prec, test_rec = evaluate_classification(model, test_loader, device)
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


def train_lstm_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = BASELINE_CLF_HYPERPARAMS['LSTM']

    model_name = 'lstm_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = LSTMClassifier(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_classes=hyperparams['num_classes'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
               'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, train_acc, train_f1, _ = train_classification_epoch(
            model, train_loader, optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        val_loss, val_acc, val_f1, _, _ = evaluate_classification(model, val_loader, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_acc': val_acc, 'val_f1': val_f1}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_acc, test_f1, test_prec, test_rec = evaluate_classification(model, test_loader, device)
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


def train_gru_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = BASELINE_CLF_HYPERPARAMS['GRU']

    model_name = 'gru_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = GRUClassifier(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_classes=hyperparams['num_classes'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                            weight_decay=hyperparams['weight_decay'])
    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
               'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        train_loss, train_acc, train_f1, _ = train_classification_epoch(
            model, train_loader, optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        val_loss, val_acc, val_f1, _, _ = evaluate_classification(model, val_loader, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={'train_loss': train_loss, 'val_acc': val_acc, 'val_f1': val_f1}
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_acc, test_f1, test_prec, test_rec = evaluate_classification(model, test_loader, device)
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency

In [ ]:
def train_min_lstm_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = MINIMAL_CLF_HYPERPARAMS['minLSTM']

    model_name = 'minlstm_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier (OPTIMIZED)")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length'],
        num_workers=0,
        cache_dir='./cache'
    )

    model = minLSTMClassifier(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_classes=hyperparams['num_classes'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=hyperparams['learning_rate'],
        weight_decay=hyperparams['weight_decay']
    )

    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'train_acc': [],
        'train_f1': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1': []
    }
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")

        train_loss, train_acc, train_f1, _ = train_classification_epoch(
            model, train_loader, optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        val_loss, val_acc, val_f1, _, _ = evaluate_classification(
            model, val_loader, device
        )
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={
                    'train_loss': train_loss,
                    'val_acc': val_acc,
                    'val_f1': val_f1
                }
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_acc, test_f1, test_prec, test_rec = evaluate_classification(
        model, test_loader, device
    )
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency


def train_min_gru_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = MINIMAL_CLF_HYPERPARAMS['minGRU']

    model_name = 'mingru_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier (OPTIMIZED)")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length'],
        num_workers=0,
        cache_dir='./cache'
    )

    model = minGRUClassifier(
        vocab_size=hyperparams['vocab_size'],
        embedding_dim=hyperparams['embedding_dim'],
        hidden_dim=hyperparams['hidden_dim'],
        num_classes=hyperparams['num_classes'],
        num_layers=hyperparams['num_layers'],
        dropout=hyperparams['dropout']
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=hyperparams['learning_rate'],
        weight_decay=hyperparams['weight_decay']
    )

    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    history = {
        'train_loss': [],
        'train_acc': [],
        'train_f1': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1': []
    }
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")

        train_loss, train_acc, train_f1, _ = train_classification_epoch(
            model, train_loader, optimizer, device, scheduler
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        val_loss, val_acc, val_f1, _, _ = evaluate_classification(
            model, val_loader, device
        )
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_manager.save_model(
                model, optimizer, hyperparams, model_name, epoch,
                metrics={
                    'train_loss': train_loss,
                    'val_acc': val_acc,
                    'val_f1': val_f1
                }
            )

    print(f"\n{'='*80}")
    print("Testing...")
    test_loss, test_acc, test_f1, test_prec, test_rec = evaluate_classification(
        model, test_loader, device
    )
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency

In [ ]:
def train_distilbert_classifier(device, hyperparams=None):
    if hyperparams is None:
        hyperparams = TRANSFORMER_CLF_HYPERPARAMS['DistilBERT']

    model_name = 'distilbert_clf'
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} Classifier")
    print(f"Hyperparameters: {hyperparams}")
    print(f"{'='*80}\n")

    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        tokenizer=tokenizer,
        batch_size=hyperparams['batch_size'],
        max_length=hyperparams['max_length']
    )

    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=4
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    optimizer = AdamW(model.parameters(), lr=hyperparams['learning_rate'],
                      weight_decay=hyperparams['weight_decay'])
    total_steps = len(train_loader) * hyperparams['num_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=hyperparams['warmup_steps'], num_training_steps=total_steps
    )

    history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
               'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0

    for epoch in range(hyperparams['num_epochs']):
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}")
        model.train()
        total_loss = 0
        all_preds = []
        all_labels = []

        for batch in tqdm(train_loader, desc="Training"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                labels=labels,
                return_dict=True
            )
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

        train_loss = total_loss / len(train_loader)
        train_acc = accuracy_score(all_labels, all_preds)
        train_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

        model.eval()
        total_val_loss = 0
        all_val_preds = []
        all_val_labels = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validating"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(
                    input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    return_dict=True
                )
                total_val_loss += outputs.loss.item()
                preds = outputs.logits.argmax(dim=1).cpu().numpy()
                all_val_preds.extend(preds)
                all_val_labels.extend(labels.cpu().numpy())

        val_loss = total_val_loss / len(val_loader)
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        val_f1 = f1_score(all_val_labels, all_val_preds, average='weighted', zero_division=0)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f"Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}\n")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_path = Path('models') / f"{model_name}_epoch{epoch}.pt"
            torch.save(model.state_dict(), model_path)
            print(f"✓ Model saved: {model_path}")

    print(f"\n{'='*80}")
    print("Testing...")
    model.eval()
    total_test_loss = 0
    all_test_preds = []
    all_test_labels = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                labels=labels,
                return_dict=True
            )
            total_test_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            all_test_preds.extend(preds)
            all_test_labels.extend(labels.cpu().numpy())

    test_loss = total_test_loss / len(test_loader)
    test_acc = accuracy_score(all_test_labels, all_test_preds)
    test_f1 = f1_score(all_test_labels, all_test_preds, average='weighted', zero_division=0)
    test_prec = precision_score(all_test_labels, all_test_preds, average='weighted', zero_division=0)
    test_rec = recall_score(all_test_labels, all_test_preds, average='weighted', zero_division=0)
    print(f"Test - Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(f" Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

    print("\nEfficiency Metrics:")
    efficiency = measure_efficiency(model, test_loader, device, num_batches=5)
    for key, val in efficiency.items():
        print(f" {key}: {val:.2f}")

    return model, history, efficiency

In [ ]:
def measure_efficiency(model, dataloader, device, num_batches=10):
    """Measure model efficiency: throughput and memory."""
    model.eval()

    # Warm up GPU
    for _ in range(2):
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            _ = model(input_ids)
            break

    torch.cuda.synchronize() if torch.cuda.is_available() else None

    # Measure throughput
    start_time = time.time()
    total_tokens = 0

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_batches:
                break

            input_ids = batch['input_ids'].to(device)
            _ = model(input_ids)
            total_tokens += input_ids.numel()

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    elapsed = time.time() - start_time

    tokens_per_sec = total_tokens / elapsed

    # Memory usage
    if torch.cuda.is_available():
        memory_mb = torch.cuda.memory_allocated() / 1e6
    else:
        memory_mb = 0

    return {
        'tokens_per_sec': tokens_per_sec,
        'memory_mb': memory_mb,
        'elapsed_time': elapsed,
        'total_tokens': total_tokens
    }


In [ ]:
def plot_training_history(histories_dict, task='lm', figsize=(14, 5)):
    if task == 'lm':
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        for model_name, history in histories_dict.items():
            axes[0].plot(history['val_loss'], label=model_name, marker='o')
            axes[1].plot(history['val_perplexity'], label=model_name, marker='s')
        axes[0].set_ylabel('Validation Loss')
        axes[1].set_ylabel('Perplexity')
        axes[0].set_title('Language Modeling: Validation Loss')
        axes[1].set_title('Language Modeling: Perplexity')
    else:  # classification
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        for model_name, history in histories_dict.items():
            axes[0].plot(history['val_acc'], label=model_name, marker='o')
            axes[1].plot(history['val_f1'], label=model_name, marker='s')
        axes[0].set_ylabel('Accuracy')
        axes[1].set_ylabel('F1-Score')
        axes[0].set_title('Classification: Validation Accuracy')
        axes[1].set_title('Classification: Validation F1-Score')

    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def compare_models_efficiency(efficiency_dict, figsize=(12, 4)):
    df = pd.DataFrame(efficiency_dict).T
    print("\nModel Efficiency Comparison:")
    print(df.to_string())

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    df['tokens_per_sec'].plot(kind='barh', ax=axes[0], color='skyblue')
    axes[0].set_title('Throughput (Tokens/sec)')
    axes[0].set_xlabel('Tokens/sec')

    df['memory_mb'].plot(kind='barh', ax=axes[1], color='lightcoral')
    axes[1].set_title('GPU Memory Usage')
    axes[1].set_xlabel('Memory (MB)')

    plt.tight_layout()
    plt.show()


In [ ]:
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

try:
    scaler = GradScaler()
    print("✓ Mixed precision (FP16) is supported!")
except:
    print("⚠ Mixed precision not available - training will use FP32")

CUDA Available: True
GPU: NVIDIA L4
GPU Memory: 23.80 GB
✓ Mixed precision (FP16) is supported!


In [ ]:
# print("\n" + "="*80)
# print("LANGUAGE MODELING BENCHMARK")
# print("="*80)

# # Insert the models to train in here
# lm_models = [
#     ('GPT-2', train_gpt2_language_model),
# ]


# lm_histories = {}
# lm_efficiencies = {}

# for model_name, train_func in lm_models:
#     try:
#         model, history, efficiency = train_func(device)
#         lm_histories[model_name] = history
#         lm_efficiencies[model_name] = efficiency
#     except Exception as e:
#         print(f"Error training {model_name}: {e}")

# if lm_histories:
#     print("\n" + "="*80)
#     print("Language Modeling Results")
#     print("="*80)
#     plot_training_history(lm_histories, task='lm')
#     compare_models_efficiency(lm_efficiencies)

In [ ]:
print("\n" + "="*80)
print("TEXT CLASSIFICATION BENCHMARK")
print("="*80)

# Insert the models to train in here
clf_models = [
    ('DistilBERT', train_distilbert_classifier)
]

clf_histories = {}
clf_efficiencies = {}

for model_name, train_func in clf_models:
    try:
        model, history, efficiency = train_func(device)
        clf_histories[model_name] = history
        clf_efficiencies[model_name] = efficiency
    except Exception as e:
        print(f"Error training {model_name}: {e}")

if clf_histories:
    print("\n" + "="*80)
    print("Classification Results")
    print("="*80)
    plot_training_history(clf_histories, task='classification')
    compare_models_efficiency(clf_efficiencies)

In [ ]:
# if lm_efficiencies:
#     lm_eff_df = pd.DataFrame(lm_efficiencies).T
#     lm_eff_df.to_csv('results/language_modeling_efficiency.csv')
#     print("\nLanguage Modeling Efficiency Results:")
#     print(lm_eff_df)

if clf_efficiencies:
    clf_eff_df = pd.DataFrame(clf_efficiencies).T
    clf_eff_df.to_csv('results/classification_efficiency.csv')
    print("\nClassification Efficiency Results:")
    print(clf_eff_df)

print("\n" + "="*80)
print("Benchmark Complete!")
print("Results saved to results/ directory")
print("Models saved to models/ directory")
print("="*80)